# Chapter 7 — Search In-Depth
## Topic 7: Reranking — Cross-Encoders (Cohere Rerank, BGE-reranker)

**Run cells in order.**

- Module 1: Setup — candidate pool with a subtle bi-encoder ranking error
- Module 2: Simulated cross-encoder reranking (offline-safe stand-in with real CrossEncoder code shown)
- Module 3: Before/after comparison — quantify what reranking actually fixed
- Module 4: Batching cost demonstration — why batched inference matters

**Install:** `pip install sentence-transformers numpy`
(For real reranking: `BAAI/bge-reranker-base` via `sentence_transformers.CrossEncoder`)


## Module 1: Setup — Candidate Pool With a Bi-Encoder Ranking Error

Builds a candidate pool where the bi-encoder (Topic 2 style) ranking has a
known flaw: a document that superficially overlaps with the query embeds
closer than the document that actually, precisely answers it.

**Requires:** nothing standalone (loads its own model)


In [ ]:
"""
MODULE 1: Setup

Candidate pool designed so a bi-encoder's coarse similarity can rank a
partially-relevant document above the precisely-relevant one -- exactly
the failure mode a cross-encoder's joint attention is meant to fix.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

CANDIDATE_POOL = [
    {"id": 0, "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
     "doc_type": "faq"},
    {"id": 1, "text": "Fixed Deposit interest rates vary by tenure and are reviewed quarterly by the bank.",
     "doc_type": "product"},
    {"id": 2, "text": "SOP Step 3: Calculate premature withdrawal penalty at 1 percent before processing closure.",
     "doc_type": "sop"},
    {"id": 3, "text": "Nominee-initiated premature withdrawal due to depositor death is exempt from the standard penalty.",
     "doc_type": "policy"},
    {"id": 4, "text": "Interest rate penalties for early loan foreclosure are separate from FD withdrawal rules.",
     "doc_type": "product"},
]

QUERY = "what is the penalty if I withdraw my FD before maturity"

print("Loading bi-encoder model (may take a moment on first run)...")
bi_encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

query_vec = bi_encoder.encode(QUERY, normalize_embeddings=True, show_progress_bar=False)
for c in CANDIDATE_POOL:
    c["embedding"] = bi_encoder.encode(c["text"], normalize_embeddings=True, show_progress_bar=False)
    c["bienc_score"] = cosine_sim(query_vec, c["embedding"])

bienc_ranked = sorted(CANDIDATE_POOL, key=lambda c: c["bienc_score"], reverse=True)

print(f"\nQuery: '{QUERY}'\n")
print("Bi-encoder ranking (Topic 2 style, independent encoding):")
for rank, c in enumerate(bienc_ranked, start=1):
    print(f"  Rank {rank} | score={c['bienc_score']:.4f} | Doc {c['id']} [{c['doc_type'].upper():<7}] {c['text'][:55]}...")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Cross-Encoder Reranking

Shows both the REAL production code pattern (`CrossEncoder` from
sentence-transformers, `BAAI/bge-reranker-base`) and a deterministic,
offline-safe simulated cross-encoder used here so the module runs without
a live model download -- clearly labeled as a stand-in.

**Requires:** Module 1


In [ ]:
"""
MODULE 2: Cross-Encoder Reranking

REAL PRODUCTION CODE (commented -- requires network access to download
BAAI/bge-reranker-base on first run):

    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder("BAAI/bge-reranker-base")
    pairs = [[QUERY, c["text"]] for c in CANDIDATE_POOL]
    scores = reranker.predict(pairs)   # ONE batched call for all pairs

SIMULATED CROSS-ENCODER (this module): a deterministic function that
approximates what joint attention would catch -- precise phrase overlap
and negative-signal detection -- so the reranking MECHANICS and PIPELINE
can be verified offline. This is NOT a substitute for the real model's
quality; it exists purely so this notebook is runnable without a live
HuggingFace download. Replace with the real CrossEncoder code above when
running with network access.
"""

def simulated_cross_encoder_score(query: str, doc_text: str) -> float:
    """Deterministic stand-in for joint query-document attention.
    Rewards precise multi-word phrase overlap (what a cross-encoder's
    attention would pick up on) more heavily than a bi-encoder's coarse
    single-vector similarity would, and penalizes distractor terms."""
    q_words = set(query.lower().split())
    d_words = set(doc_text.lower().split())
    overlap = q_words & d_words

    # reward exact bigram phrase matches -- proxy for attention picking up
    # on precise phrase-level alignment, not just bag-of-words overlap
    q_bigrams = set(zip(query.lower().split(), query.lower().split()[1:]))
    d_bigrams = set(zip(doc_text.lower().split(), doc_text.lower().split()[1:]))
    bigram_overlap = q_bigrams & d_bigrams

    score = len(overlap) * 0.15 + len(bigram_overlap) * 0.35

    # penalize documents about a DIFFERENT specific topic despite word overlap
    # (e.g. "loan foreclosure" vs "FD withdrawal" -- a cross-encoder would
    # catch this contextual mismatch far better than a bi-encoder)
    if "loan" in d_words and "loan" not in q_words:
        score -= 0.4
    if "quarterly" in d_words and "withdraw" not in d_words and "penalty" not in d_words:
        score -= 0.3

    # sigmoid-like squashing to [0, 1], matching real cross-encoder output range
    return 1 / (1 + np.exp(-score))


# -----------------------------------------------------------------------
# Batched-style application: score every candidate against the query
# -----------------------------------------------------------------------
for c in CANDIDATE_POOL:
    c["rerank_score"] = simulated_cross_encoder_score(QUERY, c["text"])

reranked = sorted(CANDIDATE_POOL, key=lambda c: c["rerank_score"], reverse=True)

print(f"Query: '{QUERY}'\n")
print("Cross-encoder (simulated) reranking:")
for rank, c in enumerate(reranked, start=1):
    print(f"  Rank {rank} | score={c['rerank_score']:.4f} | Doc {c['id']} [{c['doc_type'].upper():<7}] {c['text'][:55]}...")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Before/After Comparison — What Reranking Actually Fixed

Directly compares bi-encoder ranking (Module 1) against reranked ordering
(Module 2), highlighting rank changes and explaining WHY each moved.

**Requires:** Module 1, Module 2


In [ ]:
"""
MODULE 3: Before/After Comparison
"""

bienc_order = [c["id"] for c in bienc_ranked]
rerank_order = [c["id"] for c in reranked]

print(f"{'Doc ID':<8} | {'Bi-encoder rank':<17} | {'Reranked rank':<15} | Moved")
print("-" * 60)
for doc_id in range(len(CANDIDATE_POOL)):
    old_rank = bienc_order.index(doc_id) + 1
    new_rank = rerank_order.index(doc_id) + 1
    moved = "UP" if new_rank < old_rank else ("DOWN" if new_rank > old_rank else "same")
    print(f"{doc_id:<8} | {old_rank:<17} | {new_rank:<15} | {moved}")

print("\nTop-1 comparison:")
print(f"  Bi-encoder top-1:  Doc {bienc_order[0]} -- {next(c['text'] for c in CANDIDATE_POOL if c['id']==bienc_order[0])[:60]}...")
print(f"  Reranked top-1:    Doc {rerank_order[0]} -- {next(c['text'] for c in CANDIDATE_POOL if c['id']==rerank_order[0])[:60]}...")

if bienc_order[0] != rerank_order[0]:
    print("\nReranking CHANGED the top-1 result.")
    print("This is the core value proposition of a second reranking stage:")
    print("the bi-encoder's independent encoding missed a distinction that")
    print("joint query-document attention (even in this simplified simulation)")
    print("was able to catch.")
else:
    print("\nTop-1 unchanged this time -- reranking's value is often more visible")
    print("further down the list (rank 2-5), which is exactly what the table above shows.")

print("\nModule 3 complete. Run Module 4.")


## Module 4: Batching Cost Demonstration

Shows WHY batching every candidate pair into a single inference call
matters -- simulates the latency difference between batched and
one-at-a-time scoring using call-count as a proxy for cost.

**Requires:** Module 1


In [ ]:
"""
MODULE 4: Batching Cost Demonstration

Real cross-encoder inference cost scales with the NUMBER OF MODEL CALLS,
not just the number of pairs scored -- batching all pairs into one call
lets the underlying framework parallelize the matrix operations, while
one-call-per-pair forces full sequential overhead for every single pair.

This module counts call overhead as a proxy metric (not wall-clock time,
since the simulated scorer has negligible actual compute cost) to make
the SHAPE of the cost difference concrete.
"""

import time

N_CANDIDATES = 20  # a realistic top-K candidate pool size from Topic 4's hybrid retrieval

# Build a slightly larger synthetic pool for a clearer demonstration
synthetic_pool = [
    {"id": i, "text": f"Synthetic candidate document number {i} about FD policy details."}
    for i in range(N_CANDIDATES)
]


def score_one_at_a_time(query: str, pool: list) -> tuple:
    """Simulates calling the model once PER candidate -- the anti-pattern."""
    call_count = 0
    start = time.perf_counter()
    for c in pool:
        _ = simulated_cross_encoder_score(query, c["text"])
        call_count += 1  # one model invocation per candidate
    elapsed = time.perf_counter() - start
    return call_count, elapsed


def score_batched(query: str, pool: list) -> tuple:
    """Simulates ONE batched call scoring all candidates together --
    the correct pattern (real CrossEncoder.predict(pairs) does this)."""
    call_count = 1  # ONE model invocation for the entire batch
    start = time.perf_counter()
    _ = [simulated_cross_encoder_score(query, c["text"]) for c in pool]
    elapsed = time.perf_counter() - start
    return call_count, elapsed


one_at_a_time_calls, _ = score_one_at_a_time(QUERY, synthetic_pool)
batched_calls, _ = score_batched(QUERY, synthetic_pool)

print(f"Candidate pool size: {N_CANDIDATES}")
print(f"\nOne-at-a-time scoring: {one_at_a_time_calls} separate model invocations")
print(f"Batched scoring:       {batched_calls} single model invocation")
print(f"\nModel invocation overhead ratio: {one_at_a_time_calls / batched_calls}x")

print("""
In a real transformer model, each separate invocation carries fixed
overhead (Python/framework call overhead, GPU kernel launch overhead,
no parallelism across candidates) on top of the actual compute. Batching
lets the framework parallelize the matrix multiplications across the
batch dimension, especially significant on GPU hardware like this
project's RTX 4060 -- the wall-clock difference between one-at-a-time
and batched scoring for 20 candidates is typically far larger than the
raw compute difference alone would suggest.

Rule for this project: ALWAYS pass the full candidate list to
reranker.predict(pairs) in a single call, never loop and call
.predict() once per pair.
""")

print("=" * 70)
print("CHAPTER 7 TOPIC 7 SUMMARY")
print("=" * 70)
print("""
Cross-encoder reranking: query and document encoded JOINTLY as one input,
  letting attention directly compare tokens -- far more accurate than a
  bi-encoder's independently-computed vectors, but far too slow for
  full-corpus search.

Pipeline position: Hybrid RRF (Topic 4) -> top-20 candidates
  -> Cross-encoder reranking (this topic) -> top-5
  -> optionally MMR (Topic 6) for final diversity check
  -> passed to generation (Chapter 8)

For this project: local BGE-reranker preferred over Cohere's managed API
  due to PII/data-residency concerns for financial customer data
  (same principle as Chapter 6 Topic 9's access-scoping discussion).

Always batch candidate pairs into ONE predict() call -- demonstrated
  above with the call-count overhead comparison.

Reranking's value must be measured (Topic 9: Recall@K, NDCG@K with vs
  without reranking) against its added latency and cost, not assumed.

Next: Topic 8 -- Metadata Filtering as a Search-Time Tool
""")
